# pandas 01. 読み込みと型

`read_csv` の**引数を1つ変えると結果がどう変わるか**を見ていきます。

## このノートの進み方

各節は、こういう並びです。

```
説明 (2〜3行)
  [セルA]  ← まず実行する
  [セルB]  ← 引数を1つ変えたもの
  <details>答え合わせ</details>
  ミニ練習 (1問。すぐ答えを開いてよい)
```

セルAとBは、実行する前に「どこが変わりそうか」を一言だけ思い浮かべてから
実行すると記憶に残ります。外れて当たり前なので、気楽にどうぞ。

使うデータは `/data/orders.csv`(12行)。
わざと少し汚してあります。まずは生のまま眺めてみましょう。

In [ ]:
import io
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

# pandas を通す前の、ファイルそのままの姿
print(open("/data/orders.csv", encoding="utf-8").read())

---
## 1. 型を推測させると何が起きるか

### 1-1. 素で読む vs dtype=str

`read_csv` は、何も指定しないと列の型を推測します。
親切な機能ですが、汚れたデータでは勝手な解釈をされることがあります。

In [ ]:
# A: 素で読む
a = pd.read_csv("/data/orders.csv")
display(a.head(8))
a.dtypes

In [ ]:
# B: 全部文字列で読む
b = pd.read_csv("/data/orders.csv", dtype=str)
display(b.head(8))
b.dtypes

<details>
<summary>答え合わせ</summary>

`A` を見ると:

- `qty` が **float64**。欠損が1つあるだけで、整数の列が小数になります
- `amount` が **object**(文字列)。`"2,400"` と `N/A` が混ざっていて
  数値にできず、列ごと文字列のままです
- `order_date` も **object**。日付としては解釈されていません

`B` は全部 `object` です。「推測させない」と決めてしまう書き方で、
汚れたデータを扱うときの出発点になります。まず全部文字列で受け取って、
そのあと自分で解釈していきます。

</details>

In [ ]:
# ミニ練習: orders.csv を「全部文字列」で読む

ans = ...   # ここに書く

assert str(ans["qty"].dtype) == "object", f"object のはず: {ans['qty'].dtype}"
assert ans["amount"].iloc[6] == "2,400"
print("OK")

<details>
<summary>答え</summary>

```python
ans = pd.read_csv("/data/orders.csv", dtype=str)
```

</details>

### 1-2. IDの先頭ゼロ

推測が困るのは、こういうときです。
小さな文字列を直接 `read_csv` に読ませて確かめます。

In [ ]:
# A: 推測させる
csv = "code,name\n00123,foo\n00456,bar\n"
a = pd.read_csv(io.StringIO(csv))
display(a)
a.dtypes

In [ ]:
# B: code だけ文字列で読む
csv = "code,name\n00123,foo\n00456,bar\n"
b = pd.read_csv(io.StringIO(csv), dtype={"code": str})
display(b)
b.dtypes

<details>
<summary>答え合わせ</summary>

`A` は `00123` を整数 `123` にします。**先頭のゼロが消えます。**

一度消えると、`00123` を持っている別のテーブルと結合できません。
しかもエラーは出ないので、結合結果が0件になって初めて気づくことになります。

**IDは文字列で扱う。** これは覚えておいて損のない習慣です。

</details>

In [ ]:
# ミニ練習: 郵便番号 zip を文字列のまま読む (先頭のゼロを残す)

csv = "zip,city\n0600042,札幌\n1000001,東京\n"

ans = ...   # ここに書く

assert ans["zip"].tolist() == ["0600042", "1000001"], ans["zip"].tolist()
print("OK")

<details>
<summary>答え</summary>

```python
ans = pd.read_csv(io.StringIO(csv), dtype={"zip": str})
```

</details>

---
## 2. 欠損をどう受け取るか

### 2-1. keep_default_na

pandas は、いくつかの文字列を既定で欠損(`NaN`)として読みます。
何が対象になるのかを見ておきましょう。

In [ ]:
# A: 既定 (keep_default_na=True)
a = pd.read_csv("/data/orders.csv", dtype=str)
display(a[["order_id", "customer_id", "amount", "qty"]])
a.isna().sum()

In [ ]:
# B: 何も欠損扱いしない
b = pd.read_csv("/data/orders.csv", dtype=str, keep_default_na=False)
display(b[["order_id", "customer_id", "amount", "qty"]])
b.isna().sum()

<details>
<summary>答え合わせ</summary>

`A` では `N/A` と空欄が `NaN` になっています。
既定の欠損リストには `NA` `N/A` `NULL` `NaN` `None` など約20種が入っています
(全部見たいときは `pd._libs.parsers.STR_NA_VALUES`)。

`B` は**何も欠損にしません**。`N/A` は文字列 `'N/A'` のまま、空欄は `''` のままです。

どちらが良いかは場合によります。ただし `A` は「pandas が知っている表現」しか拾わず、
`-` や `不明` や `999` は素通りします。結局あとで自分で判定するなら、
最初から `B` にして、欠損の定義を1箇所にまとめたほうが読みやすくなります。

</details>

In [ ]:
# ミニ練習: keep_default_na=False で読むと、amount の "N/A" は文字列のまま残る。
#           "N/A" が何件あるかを数える

b = pd.read_csv("/data/orders.csv", dtype=str, keep_default_na=False)

ans = ...   # ここに書く

assert ans == 1, f"1件のはず: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = (b["amount"] == "N/A").sum()
```

</details>

### 2-2. na_values で足す vs 自分で判定する

そのデータ固有の欠損表現(`-` や `不明` など)を、どこで吸収するかの話です。

In [ ]:
# A: read_csv に教える
a = pd.read_csv("/data/orders.csv", dtype=str,
                keep_default_na=False, na_values=["N/A", "-", ""])
display(a[["customer_id", "amount"]])
a.isna().sum()

In [ ]:
# B: 読んだあとで置き換える
NULLISH = {"NULL", "N/A", "-", ""}
b = pd.read_csv("/data/orders.csv", dtype=str, keep_default_na=False)
b = b.map(lambda s: None if str(s).strip() in NULLISH else s)
display(b[["customer_id", "amount"]])
b.isna().sum()

<details>
<summary>答え合わせ</summary>

結果は同じです。違うのは**吸収する場所**です。

- `A` は読み込み時に片付きます。短いですが、ファイルごとに `read_csv` の引数が散らばります
- `B` は「欠損の定義」がコードの中に1つだけあります。前後の空白も落とせます
  (`" N/A "` は `A` では拾えません)

複数ファイルを読むなら `B` のほうが揃えやすいです。
`DataFrame.map` は全セルに関数をかけます(pandas 2.1 以前は `applymap`)。

</details>

In [ ]:
# ミニ練習: 下の csv を読むとき、"不明" も欠損として読む

csv = "name,tier\n佐藤,gold\n鈴木,不明\n"

ans = ...   # ここに書く

assert ans["tier"].isna().sum() == 1, ans["tier"].tolist()
print("OK")

<details>
<summary>答え</summary>

```python
ans = pd.read_csv(io.StringIO(csv), na_values=["不明"])
```

</details>

---
## 3. 数値に変える

### 3-1. astype vs to_numeric(errors=)

文字列を数値にする書き方が2つあります。失敗したときの動きが違います。

In [ ]:
# A: astype
s = pd.Series(["1200", "980", "N/A", "2,400", "0"])
try:
    print(s.astype(int))
except Exception as e:
    print(f"{type(e).__name__}: {e}")

In [ ]:
# B: to_numeric(errors='coerce')
s = pd.Series(["1200", "980", "N/A", "2,400", "0"])
print(pd.to_numeric(s, errors="coerce"))

<details>
<summary>答え合わせ</summary>

- `A` は**落ちます**。最初に失敗した値を教えてくれます
- `B` は失敗した値を**黙って `NaN`** にします。落ちない代わりに、
  何件が欠損になったかは自分で数えないと気づけません

`"2,400"` は `B` でも `NaN` になっている点に注目してください。
**カンマは自動では外れません。** `errors="coerce"` は「読めないものを捨てる」だけで、
「読めるように直す」わけではありません。

使い分けはこう考えると迷いません。

- 変換できないものが**あってはならない** → `astype`(落ちてほしい)
- 変換できないものが**あると分かっている** → `to_numeric(errors="coerce")` のあと、
  `isna().sum()` で件数を必ず見る

</details>

In [ ]:
# ミニ練習: 下の Series を数値にする。変換できないものは NaN にして、その件数を数える

s = pd.Series(["100", "abc", "300", ""])

ans = ...   # ここに書く

assert ans == 2, f"2件のはず: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = pd.to_numeric(s, errors="coerce").isna().sum()
```

</details>

### 3-2. thousands= で読む

`"2,400"` のような桁区切りは、`read_csv` 側で外すこともできます。

In [ ]:
# A: thousands なし
csv = "amount\n1200\n\"2,400\"\n980\n"
a = pd.read_csv(io.StringIO(csv))
print(a.dtypes)
a

In [ ]:
# B: thousands=','
csv = "amount\n1200\n\"2,400\"\n980\n"
b = pd.read_csv(io.StringIO(csv), thousands=",")
print(b.dtypes)
b

<details>
<summary>答え合わせ</summary>

`B` は `int64` になります。`thousands=","` が桁区切りのカンマを外してくれます。

ただし効くのは「その列全体が数値として解釈できる」ときだけです。
`N/A` が混ざると `object` に戻ります。実データでは `thousands` だけで
片付くことは少なく、結局は自分で正規化することが多いです。

引数で済むなら引数で、済まないなら自分で書く。
どちらでもよいですが、**両方書いて二重に処理しない**ようにだけ気をつけます。

</details>

In [ ]:
# ミニ練習: 下の csv を、price が int になるように読む

csv = "item,price\nコーヒー,\"1,200\"\nケーキ,\"3,400\"\n"

ans = ...   # ここに書く

assert str(ans["price"].dtype).startswith("int"), ans["price"].dtype
assert ans["price"].sum() == 4600
print("OK")

<details>
<summary>答え</summary>

```python
ans = pd.read_csv(io.StringIO(csv), thousands=",")
```

</details>

---
## 4. 整数と欠損

### 4-1. int64 と Int64

欠損を含む整数の列をどう持つか、という話です。
大文字で始まる `Int64` は、`int64` とは別の型です。

In [ ]:
# A: 既定の型
a = pd.read_csv("/data/orders.csv")
print(a["qty"].dtype)
display(a["qty"].head(6))
print("合計:", a["qty"].sum())

In [ ]:
# B: 欠損を持てる整数型
b = pd.read_csv("/data/orders.csv")
b["qty"] = b["qty"].astype("Int64")
print(b["qty"].dtype)
display(b["qty"].head(6))
print("合計:", b["qty"].sum())

<details>
<summary>答え合わせ</summary>

`A` は `float64` です。欠損があるので整数のままではいられません。
表示が `2.0` `1.0` になり、Parquet に書けば小数の列になります。

`B` は `Int64`(先頭大文字)。pandas の **nullable 整数型**で、
欠損を `<NA>` として持ちながら整数でいられます。

| | 欠損 | 表示 | 由来 |
| --- | --- | --- | --- |
| `int64` | 持てない | `2` | NumPy |
| `float64` | `NaN` | `2.0` | NumPy |
| `Int64` | `<NA>` | `2` | pandas |

どちらも `sum()` は欠損を無視します。

なお `astype("int64")` は欠損があると落ちるので、
整数にしたいなら先に欠損を埋めるか除くかを決めておきます。

</details>

In [ ]:
# ミニ練習: 下の Series を、欠損を持てる整数型にする

s = pd.Series([1.0, np.nan, 3.0])

ans = ...   # ここに書く

assert str(ans.dtype) == "Int64", ans.dtype
assert ans.tolist()[0] == 1
print("OK")

<details>
<summary>答え</summary>

```python
ans = s.astype("Int64")
```

</details>

---
## 5. 日付に変える

### 5-1. format を指定するかどうか

`to_datetime` は書式を推測できますが、指定したほうが安全です。

In [ ]:
# A: 推測させる
s = pd.Series(["2024-01-05", "2024-01-06", ""])
print(pd.to_datetime(s, errors="coerce"))

In [ ]:
# B: 書式を明示する
s = pd.Series(["2024-01-05", "2024-01-06", ""])
print(pd.to_datetime(s, format="%Y-%m-%d", errors="coerce"))

<details>
<summary>答え合わせ</summary>

この例では同じ結果です。差が出るのは**曖昧な書式**のときです。

```python
pd.to_datetime(pd.Series(["01/05/2024", "13/05/2024"]), errors="coerce")
```

`01/05/2024` は1月5日か5月1日か決まりません。pandas は列の中身から
推測しようとするので、**ファイルによって解釈が変わる**ことがあります。
書式が分かっているなら `format=` を書いておくと安心です(そのほうが速くもあります)。

`errors="coerce"` は変換できないものを `NaT` にします。
`NaT` は「日時の欠損」で、`isna()` で判定できます。

</details>

In [ ]:
# ミニ練習: "2024/01/05" の形式を、format を指定して日付にする

s = pd.Series(["2024/01/05", "2024/01/06"])

ans = ...   # ここに書く

assert str(ans.dtype).startswith("datetime64"), ans.dtype
assert str(ans.iloc[0].date()) == "2024-01-05"
print("OK")

<details>
<summary>答え</summary>

```python
ans = pd.to_datetime(s, format="%Y/%m/%d")
```

</details>

### 5-2. datetime64 と date

日付だけが欲しいのに時刻まで付いてくる、という場面の話です。

In [ ]:
# A: datetime64 のまま
s = pd.to_datetime(pd.Series(["2024-01-05", "2024-01-06"]))
print(s.dtype)
print(s.tolist())

In [ ]:
# B: date にする
s = pd.to_datetime(pd.Series(["2024-01-05", "2024-01-06"]))
d = s.dt.date
print(d.dtype)
print(d.tolist())

<details>
<summary>答え合わせ</summary>

`A` は `datetime64[ns]` で、中身は `Timestamp`(時刻を持ちます)。
`B` は `object` で、中身は `datetime.date`(日付だけ)です。

Parquet に「日付型(`date32`)」で書きたいときは `B` の形が要ります。
一方、日付の差を取ったり月で丸めたりする計算は `A` のほうがやりやすいです。

**計算している間は `datetime64`、書き出す直前に `.dt.date`**、と覚えておけば十分です。

</details>

In [ ]:
# ミニ練習: orders.csv の order_date を日付にして、時刻を落とした Series を作る
#           (欠損は欠損のまま)

o = pd.read_csv("/data/orders.csv")

ans = ...   # ここに書く

assert ans.dropna().map(lambda v: type(v).__name__).eq("date").all()
assert ans.isna().sum() == 1
print("OK")

<details>
<summary>答え</summary>

```python
ans = pd.to_datetime(o["order_date"], format="%Y-%m-%d", errors="coerce").dt.date
```

</details>

---
## 6. 欠損の比較

### 6-1. NaN は自分自身と等しくない

欠損の判定に `==` を使わない理由です。

In [ ]:
# A: == で比べる
print(np.nan == np.nan)
s = pd.Series([1.0, np.nan, 3.0])
print(s == np.nan)

In [ ]:
# B: isna で判定する
s = pd.Series([1.0, np.nan, 3.0])
print(s.isna())
print("欠損の数:", s.isna().sum())

<details>
<summary>答え合わせ</summary>

`NaN == NaN` は **False** です。これは浮動小数点の仕様で、SQL の NULL も同じ考え方です。
そのため `df[df["col"] == np.nan]` は**必ず0件**になります。エラーは出ません。

欠損の判定は `isna()` / `notna()` を使います。

欠損の実体は文脈によって呼び名が変わります。

| | どこで出るか |
| --- | --- |
| `np.nan` | float の列 |
| `None` | object の列 |
| `pd.NaT` | datetime の列 |
| `pd.NA` | nullable な型 (`Int64` など) |

`isna()` はどれも `True` にしてくれるので、**判定はいつも `isna()`** で大丈夫です。

</details>

In [ ]:
# ミニ練習: 欠損でない行だけを取り出す

s = pd.Series([1.0, np.nan, 3.0, np.nan])

ans = ...   # ここに書く

assert ans.tolist() == [1.0, 3.0], ans.tolist()
print("OK")

<details>
<summary>答え</summary>

```python
ans = s[s.notna()]
```

</details>

---
## 仕上げ

ここまでの引数を組み合わせます。少し長いので、答えを見ながらでも大丈夫です。

In [ ]:
# 仕上げ1: orders.csv を「型を推測させずに」読み、
#          NULL / N/A / - / 空文字 (前後に空白があっても) を欠損にする

ans = ...   # ここに書く

assert ans["amount"].isna().sum() == 1, f"amount の欠損は1件: {ans['amount'].isna().sum()}"
assert ans["qty"].isna().sum() == 1
assert ans["customer_id"].isna().sum() == 1
assert ans["order_date"].isna().sum() == 1
assert ans["amount"].dtype == object, "この時点ではまだ文字列"
print("OK")

<details>
<summary>答え</summary>

```python
NULLISH = {"NULL", "N/A", "-", ""}
ans = pd.read_csv("/data/orders.csv", dtype=str, keep_default_na=False)
ans = ans.map(lambda s: None if str(s).strip() in NULLISH else s)
```

</details>

In [ ]:
# 仕上げ2: 仕上げ1の ans の amount を整数にする。
#          "2,400" のようなカンマ区切りも通し、欠損は欠損のまま残す。
#          (ヒント: 欠損を含む整数列は Int64)

amount = ...   # ここに書く (Series)

assert str(amount.dtype) == "Int64", f"Int64 のはず: {amount.dtype}"
assert amount.isna().sum() == 1
assert amount.sum() == 23220, f"合計が違う: {amount.sum()}"
print("OK")

<details>
<summary>答え</summary>

```python
amount = pd.to_numeric(
    ans["amount"].str.replace(",", "", regex=False), errors="coerce"
).astype("Int64")
```

</details>

In [ ]:
# 仕上げ3: 仕上げ1の ans の order_date を date にする。欠損は欠損のまま。

order_date = ...   # ここに書く (Series)

assert order_date.isna().sum() == 1
non_null = order_date.dropna()
assert all(type(v).__name__ == "date" for v in non_null), "datetime.date のはず"
assert str(non_null.iloc[0]) == "2024-01-05"
print("OK")

<details>
<summary>答え</summary>

```python
order_date = pd.to_datetime(
    ans["order_date"], format="%Y-%m-%d", errors="coerce"
).dt.date
```

</details>

---
## まとめ

| 引数 / 書き方 | 効果 | いつ使うか |
| --- | --- | --- |
| `dtype=str` | 型を推測させない | 汚れたデータを扱うとき |
| `dtype={"id": str}` | 列を名指しで文字列に | IDの先頭ゼロを守る |
| `keep_default_na=False` | 勝手に欠損にしない | 欠損の定義を自分で持つとき |
| `na_values=[...]` | 欠損表現を足す | 読み込み時に片付けたいとき |
| `thousands=","` | 桁区切りを外す | 数値列が単純に汚れているだけのとき |
| `astype(int)` | 失敗したら落ちる | 失敗があってはならないとき |
| `to_numeric(errors="coerce")` | 失敗を NaN にする | 失敗を想定し、件数を数えるとき |
| `Int64` | 欠損を持てる整数 | 途中の計算で整数を保ちたいとき |
| `to_datetime(format=...)` | 推測させない | 書式が分かっているとき |
| `.dt.date` | 時刻を落とす | 日付として書き出す直前 |
| `isna()` | 欠損の判定 | `== np.nan` の代わりに、いつも |

次は `pandas-02-select-and-transform.ipynb` です。